
# Phase 8B — Context Signal Audit & Scoring Redesign

This notebook addresses the degeneracy discovered in Phase 10:

\[
F(S)\in\{0,1\}
\quad\text{and}\quad
F(S)=\text{constant for all }16\text{ masks within a task.}
\]

The goal is to determine **why**:

1. the model already knows the answer without context;
2. `mask=0` accidentally leaks evidence;
3. the binary scorer hides confidence changes;
4. a task is too easy / too deterministic;
5. subset construction is not actually changing the prompt.

The notebook therefore audits the **raw prompt/evidence construction** and produces richer subset-level signals.

Recommended output:

\[
Q_{\text{binary}}(S),\quad
Q_{\text{replicated}}(S),\quad
Q_{\text{continuous}}(S)
\]

where the latter may be a log-probability, margin, confidence, or another continuous score when the provider exposes one.


In [1]:

import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)



## 1. Load the Phase-8 raw table

The notebook is intentionally schema-tolerant.  
It looks for common column names and reports what is available.

For best results, the raw table should contain one row per `(task_id, mask, repeat)` and, if possible:

- `provider`
- `task_id`
- `family`
- `mask`
- `prompt` or rendered input
- `evidence` / `context`
- `answer`
- `gold`
- `correct`
- optional `logprob_gold`, `margin`, `confidence`
- optional `repeat_id`


In [8]:

# Set this to your exported Phase-8 raw CSV / parquet / JSONL.
DATA_PATH = "phase8B_rag_raw_results.csv"

def load_table(path):
    p = Path(path)
    suf = p.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(p)
    if suf in {".parquet", ".pq"}:
        return pd.read_parquet(p)
    if suf in {".jsonl", ".ndjson"}:
        return pd.read_json(p, lines=True)
    if suf == ".json":
        return pd.read_json(p)
    raise ValueError(f"Unsupported extension: {suf}")

if DATA_PATH is None:
    print("Set DATA_PATH to load real Phase-8 data.")
    raw = None
else:
    raw = load_table(DATA_PATH)
    print("rows:", len(raw))
    print("columns:", list(raw.columns))
    display(raw.head())


rows: 256
columns: ['run_id', 'manifest_sha256', 'provider', 'requested_model', 'returned_model', 'query_id', 'dataset_index', 'question', 'gold_answer', 'mask', 'order', 'repeat', 'effort', 'raw_text', 'exact_match', 'token_f1', 'quality', 'context_token_proxy', 'input_tokens', 'output_tokens', 'latency_sec', 'provider_cost_usd', 'error']


,run_id,manifest_sha256,provider,requested_model,returned_model,query_id,dataset_index,question,gold_answer,mask,order,repeat,effort,raw_text,exact_match,token_f1,quality,context_token_proxy,input_tokens,output_tokens,latency_sec,provider_cost_usd,error
0,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,0,0,0,low,UNKNOWN,0.0,0.00,0.00,0,102,2,75.856878,0.0,NaN
1,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,1,1,0,low,Mountain Valley High School,0.0,0.00,0.00,41,150,5,113.243740,0.0,NaN
2,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,2,1,0,low,Mountain Valley High School,0.0,0.00,0.00,80,182,5,134.191197,0.0,NaN
3,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,3,2,0,low,"The main industry of the town is a papermaking industry, and the school is located in Metea Valley.",0.0,0.25,0.25,121,234,23,187.184164,0.0,NaN
4,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,4,1,0,low,Mountain Valley High School,0.0,0.00,0.00,89,190,5,144.571601,0.0,NaN


In [9]:

ALIASES = {
    "task": ["task_id", "qid", "query_id", "id"],
    "mask": ["mask", "subset_mask", "context_mask"],
    "family": ["family", "task_family", "kind"],
    "provider": ["provider", "model", "backend"],
    "prompt": ["prompt", "rendered_prompt", "input_text", "request"],
    "context": ["context", "evidence", "selected_evidence", "support_text"],
    "gold": ["gold", "target", "reference_answer", "answer_gold"],
    "answer": ["answer", "prediction", "response", "model_answer"],
    "correct": ["correct", "is_correct", "success"],
    "repeat": ["repeat_id", "repeat", "sample_id", "trial"],
    "logprob": ["logprob_gold", "gold_logprob", "target_logprob"],
    "margin": ["margin", "gold_margin"],
    "confidence": ["confidence", "gold_confidence", "prob_gold"],
}

def resolve_col(df, logical_name):
    for c in ALIASES[logical_name]:
        if c in df.columns:
            return c
    return None

def resolved_schema(df):
    return {k: resolve_col(df, k) for k in ALIASES}

if raw is not None:
    schema = resolved_schema(raw)
    display(pd.DataFrame([schema]))


,task,mask,family,provider,prompt,context,gold,answer,correct,repeat,logprob,margin,confidence
0,query_id,mask,None,provider,None,None,None,None,None,repeat,None,None,None



## 2. Mask coverage and within-task variation

This is the first hard check.

For \(n=4\), every complete task should have 16 unique masks:

\[
0,\dots,15.
\]

We also compute whether `correct` is constant across all masks.


In [10]:

def mask_variation_report(df, n_evidence=4):
    schema = resolved_schema(df)
    task_col = schema["task"]
    mask_col = schema["mask"]
    correct_col = schema["correct"]
    provider_col = schema["provider"]
    family_col = schema["family"]

    if task_col is None or mask_col is None:
        raise ValueError("Need task_id-like and mask-like columns.")

    group_cols = [c for c in [provider_col, task_col, family_col] if c is not None]

    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        masks = pd.to_numeric(g[mask_col], errors="coerce").dropna().astype(int)
        row["n_rows"] = len(g)
        row["n_masks"] = masks.nunique()
        row["complete_cube"] = masks.nunique() == 2**n_evidence
        row["mask_min"] = masks.min() if len(masks) else np.nan
        row["mask_max"] = masks.max() if len(masks) else np.nan

        if correct_col is not None:
            x = pd.to_numeric(g[correct_col], errors="coerce").dropna().to_numpy(float)
            row["correct_mean"] = float(x.mean()) if len(x) else np.nan
            row["correct_std"] = float(x.std(ddof=0)) if len(x) else np.nan
            row["correct_unique"] = int(len(np.unique(x))) if len(x) else 0
            row["correct_flat"] = bool(len(x) and np.ptp(x) < 1e-12)
        rows.append(row)

    return pd.DataFrame(rows)

if raw is not None:
    mask_report = mask_variation_report(raw, n_evidence=4)
    display(mask_report)


,provider,query_id,n_rows,n_masks,complete_cube,mask_min,mask_max
0,local,5a855b5c5542997b5ce3ffeb,64,64,False,0,63
1,local,5a860aa6554299211dda2a45,64,64,False,0,63
2,local,5a90b1a15542990a984936ab,64,64,False,0,63
3,local,5abcd77755429965836004ce,64,64,False,0,63



## 3. Prompt/context leakage audit

For each task we compare `mask=0` against `mask=full`.

We flag:
- identical rendered prompts;
- identical context strings;
- answer/gold string accidentally present in the zero-mask prompt;
- evidence markers still present in the zero-mask prompt.

This does not prove semantic leakage, but catches common construction errors.


In [11]:

def normalize_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def leakage_audit(df, n_evidence=4):
    schema = resolved_schema(df)
    task_col = schema["task"]
    mask_col = schema["mask"]
    prompt_col = schema["prompt"]
    context_col = schema["context"]
    gold_col = schema["gold"]
    provider_col = schema["provider"]

    if task_col is None or mask_col is None:
        raise ValueError("Need task and mask columns.")

    full_mask = 2**n_evidence - 1
    group_cols = [c for c in [provider_col, task_col] if c is not None]
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))

        g0 = g[pd.to_numeric(g[mask_col], errors="coerce") == 0]
        gf = g[pd.to_numeric(g[mask_col], errors="coerce") == full_mask]

        row["has_mask0"] = len(g0) > 0
        row["has_full"] = len(gf) > 0

        if prompt_col and len(g0) and len(gf):
            p0 = normalize_text(g0.iloc[0][prompt_col])
            pf = normalize_text(gf.iloc[0][prompt_col])
            row["prompt0_equals_promptfull"] = (p0 == pf)
            row["prompt0_len"] = len(p0)
            row["promptfull_len"] = len(pf)
        else:
            row["prompt0_equals_promptfull"] = np.nan

        if context_col and len(g0) and len(gf):
            c0 = normalize_text(g0.iloc[0][context_col])
            cf = normalize_text(gf.iloc[0][context_col])
            row["context0_empty"] = (c0 == "")
            row["context0_equals_contextfull"] = (c0 == cf)
            row["context0_len"] = len(c0)
            row["contextfull_len"] = len(cf)

        if gold_col and prompt_col and len(g0):
            gold = normalize_text(g0.iloc[0][gold_col]).lower()
            p0 = normalize_text(g0.iloc[0][prompt_col]).lower()
            row["gold_literal_in_mask0_prompt"] = bool(gold and gold in p0)

        rows.append(row)

    return pd.DataFrame(rows)

if raw is not None:
    leak = leakage_audit(raw, n_evidence=4)
    display(leak)


,provider,query_id,has_mask0,has_full,prompt0_equals_promptfull
0,local,5a855b5c5542997b5ce3ffeb,True,True,NaN
1,local,5a860aa6554299211dda2a45,True,True,NaN
2,local,5a90b1a15542990a984936ab,True,True,NaN
3,local,5abcd77755429965836004ce,True,True,NaN



## 4. Richer \(F(S)\): aggregate repeats and continuous scores

Binary correctness is too coarse if a model's confidence changes without changing the final answer.

For repeated sampling:

\[
\hat p_{\text{correct}}(S)
=
\frac1R\sum_{r=1}^R \mathbf 1[\hat y_r=y^\star].
\]

For continuous provider scores, prefer:
- gold log-probability;
- gold-vs-runner-up margin;
- calibrated confidence.

The function below builds a subset surface from whichever signals exist.


In [12]:

def build_subset_surfaces(df):
    schema = resolved_schema(df)
    task_col = schema["task"]
    mask_col = schema["mask"]
    provider_col = schema["provider"]
    family_col = schema["family"]

    group_cols = [c for c in [provider_col, task_col, family_col, mask_col] if c is not None]
    named_aggs = {}

    if schema["correct"] is not None:
        named_aggs["p_correct"] = (schema["correct"], "mean")
        named_aggs["n_repeats"] = (schema["correct"], "size")

    if schema["logprob"] is not None:
        named_aggs["mean_logprob_gold"] = (schema["logprob"], "mean")

    if schema["margin"] is not None:
        named_aggs["mean_margin"] = (schema["margin"], "mean")

    if schema["confidence"] is not None:
        named_aggs["mean_confidence"] = (schema["confidence"], "mean")

    if not named_aggs:
        raise ValueError("No usable scoring column found.")

    out = df.groupby(group_cols, dropna=False).agg(**named_aggs).reset_index()
    return out

if raw is not None:
    surfaces = build_subset_surfaces(raw)
    display(surfaces.head(20))


ValueError: No usable scoring column found.


## 5. Endpoint contrast: does evidence matter at all?

For every signal \(F\):

\[
\Delta_{\text{full-empty}}
=
F(\text{full})-F(\emptyset).
\]

If this is approximately zero for nearly every task, do not proceed to spectral interpretation yet.


In [7]:

def endpoint_contrast(surface_df, signal_col, n_evidence=4):
    schema = resolved_schema(surface_df)
    task_col = schema["task"] or "task_id"
    mask_col = schema["mask"] or "mask"
    provider_col = schema["provider"]
    family_col = schema["family"]

    full_mask = 2**n_evidence - 1
    group_cols = [c for c in [provider_col, task_col, family_col] if c is not None]

    rows = []
    for keys, g in surface_df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        g0 = g[pd.to_numeric(g[mask_col], errors="coerce") == 0]
        gf = g[pd.to_numeric(g[mask_col], errors="coerce") == full_mask]
        if len(g0) and len(gf):
            f0 = float(g0.iloc[0][signal_col])
            ff = float(gf.iloc[0][signal_col])
            row.update({
                "empty": f0,
                "full": ff,
                "full_minus_empty": ff-f0,
            })
        rows.append(row)
    return pd.DataFrame(rows)

# Example:
# contrasts = endpoint_contrast(surfaces, "p_correct", 4)
# display(contrasts)



## 6. Decision gate

Proceed to Phase 10 only if at least one of these is true for a meaningful fraction of tasks:

\[
\operatorname{std}_S F(S)>0
\]

or

\[
F(\text{full})-F(\emptyset)\neq0.
\]

Otherwise redesign the tasks or scoring before spectral analysis.
